In [ ]:
%run "../utils/00_config"

In [ ]:
# Runtime parameters
dbutils.widgets.text(
    "products_file",
    "",
    "Selected products file"
)

WFM_USER_AGENT = dbutils.secrets.get(
    scope="warframe-market",
    key="wfm_user_agent"
)

PRODUCTS_FILE = dbutils.widgets.get("products_file").strip()

if not PRODUCTS_FILE:
    raise ValueError("The selected products file path cannot be empty.")

if not WFM_USER_AGENT or "configure-your-contact" in WFM_USER_AGENT.lower():
    raise ValueError(
        "Configure a valid Warframe Market User-Agent before running the collector."
    )

print(f"Environment: {ENV}")
print(f"Raw output path: {RAW_PATH}")
print(f"Products configuration: {PRODUCTS_FILE}")

In [ ]:
#Imports and environment setup
import logging
import os
from pathlib import Path

from primo_market.api import WarframeMarketClient
from primo_market.collector import collect_market
from primo_market.settings import Settings, load_selected_products
from primo_market.storage import save_execution

# Configure package environment
os.environ["SELECTED_PRODUCTS_FILE"] = PRODUCTS_FILE
os.environ["OUTPUT_DIR"] = RAW_PATH
os.environ["WFM_USER_AGENT"] = WFM_USER_AGENT

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

logger = logging.getLogger("warframe_market_collector")

print("Collector dependencies loaded successfully.")

In [ ]:
# Load runtime settings
settings = Settings.from_env()

# Validate output directory
output_path = Path(RAW_PATH)
output_path.mkdir(parents=True, exist_ok=True)

if not output_path.exists():
    raise FileNotFoundError(f"Raw output directory not found: {RAW_PATH}")

print(f"API base URL: {settings.base_url}")
print(f"Language: {settings.language}")
print(f"Requests per second: {settings.requests_per_second}")
print(f"Request timeout: {settings.timeout_seconds}s")
print(f"Maximum retries: {settings.max_retries}")
print(f"Output directory: {output_path}")

In [ ]:
# Load enabled products from YAML
products = load_selected_products(settings.selected_products_file)

if not products:
    raise ValueError("No enabled products were found.")

print(f"Selected products: {len(products)}")

for product in products:
    print(
        f"- {product['product_name']} "
        f"({product['product_slug']})"
    )

In [ ]:
# Initialize Warframe Market API client
client = WarframeMarketClient(
    base_url=settings.base_url,
    user_agent=settings.user_agent,
    language=settings.language,
    requests_per_second=settings.requests_per_second,
    timeout_seconds=settings.timeout_seconds,
    max_retries=settings.max_retries
)

logger.info("Warframe Market API client initialized.")

In [ ]:
# Collect catalog and market orders
logger.info("Starting Warframe Market collection.")

bundle = collect_market(
    client=client,
    products=products
)

logger.info(
    "Collection completed: execution_id=%s, status=%s",
    bundle["execution_id"],
    bundle["status"]
)

print(f"Execution ID: {bundle['execution_id']}")
print(f"Execution status: {bundle['status']}")
print(f"API responses: {len(bundle['events'])}")
print(f"Items processed: {len(bundle['item_execution'])}")

In [ ]:
# Persist raw API responses, item execution log and manifest
execution_path = save_execution(
    bundle=bundle,
    output_dir=RAW_PATH
)

logger.info("Raw execution persisted at: %s", execution_path)

print(f"Execution path: {execution_path}")

In [ ]:
# Validate persisted execution
manifest_path = execution_path / "manifest.json"
raw_responses_path = execution_path / "raw_api_responses.jsonl.gz"
item_execution_path = execution_path / "item_execution.jsonl.gz"

required_files = [
    manifest_path,
    raw_responses_path,
    item_execution_path
]

missing_files = [
    str(path) for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing execution files: {missing_files}"
    )

print("All execution files were successfully persisted.")

In [ ]:
# Build execution summary
successful_items = sum(
    item["status"] == "success"
    for item in bundle["item_execution"]
)

failed_items = sum(
    item["status"] == "failed"
    for item in bundle["item_execution"]
)

total_orders = sum(
    item["order_count"] or 0
    for item in bundle["item_execution"]
)

summary = {
    "execution_id": bundle["execution_id"],
    "status": bundle["status"],
    "started_at": bundle["started_at"],
    "finished_at": bundle["finished_at"],
    "api_responses": len(bundle["events"]),
    "items_processed": len(bundle["item_execution"]),
    "successful_items": successful_items,
    "failed_items": failed_items,
    "total_orders": total_orders,
    "output_path": str(execution_path)
}

for key, value in summary.items():
    print(f"{key}: {value}")

# Fail the task after persistence if collection was partial
if bundle["status"] != "success":
    raise RuntimeError(
        f"Collection finished with status '{bundle['status']}'. "
        f"{failed_items} item(s) failed. "
        f"Partial results were persisted at {execution_path}."
    )

logger.info("Warframe Market collection finished successfully.")